# Quadwind · One Recurrent PPO update by hand
**Can one short flight change the next command?** We will collect three decisions, work back from their rewards, predict the change to one actor weight and measure what happened to both the policy and the critic.

**Read the training loop in four lines.** Each name is defined below. Run the prepared cells and use their tables to explain one flight.

```python
rollout = collector.collect(3, gamma, gae_lambda)
batch = next(rollout.minibatches(3, 3, rng))
loss = ppo_loss(policy, batch)
optimizer_step(policy, optimizer, loss)
```

For the live explanation, point to one memory value, one advantage and the weight before and after the update. The gradient derivation in section 5 can wait for questions.

This notebook calls the same [recurrent actor and critic, loss and optimizer step](../quadwind/native_policy.py) and [rollout collector, GAE and sequence minibatches](../quadwind/native_rollout.py) as the project's [PyTorch training loop](../quadwind/native_training.py). The flight here uses just one axis and a single memory cell per network so we can check the arithmetic. **Run all cells in order** from the project checkout. The example runs on CPU with a fixed seed and requires no flight checkpoint.

## 1. What does the agent observe?

Our three-decision task starts $0.4$ units from the target $x=0$. A constant wind $w=0.2$ pushes toward positive $x$. The actor sees $x$. The critic sees both $x$ and $w$ during training. A negative command pushes back against the wind. The signed command is $u=\mathrm{clip}(a,-1,1)$, where $a$ is sampled from the actor's Gaussian policy. Each step gives

$$x_{t+1}=x_t+0.2(u_t+w),\qquad r_t=1-|x_{t+1}|-0.05 u_t^2.$$

The third reward ends this finite task. Its continuation value is zero. On a full Quadwind flight a time-limit cutoff can also end a recording. In that case the collector bootstraps from the final observation before reset.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import sys

import numpy as np
import torch

# Jupyter may start in the repository root or in notebook/.
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "quadwind").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from quadwind.native_policy import NativeActorCritic, optimizer_step, ppo_loss
from quadwind.native_rollout import RolloutCollector

torch.set_num_threads(1)


class ThreeDecisionHover:
    """One vector-environment slot, with the same reset/step interface as Quadwind."""
    num_envs = 1
    wind = 0.2
    action_space = SimpleNamespace(low=np.array([-1.0]), high=np.array([1.0]))

    def observe(self):
        return np.array([[self.x, self.wind]], dtype=np.float32)

    def reset(self):
        self.x, self.t = 0.4, 0
        return self.observe()

    def step(self, commands):
        u = float(commands[0, 0])
        self.x += 0.2 * (u + self.wind)
        self.t += 1
        reward = 1.0 - abs(self.x) - 0.05 * u**2
        done = self.t == 3
        obs = self.observe()
        info = {"terminal_observation": obs[0].copy()} if done else {}
        if done:
            obs = self.reset()
        return obs, np.array([reward]), np.array([done]), [info]

### A readable LSTM

Both networks have one hidden unit. For the actor, choose input, forget and output gate values $i=f=o=0.5$ by setting their weights and biases to zero. Only the candidate cell has nonzero input and recurrent weights. Then

$$g_t=\tanh(x_t+0.5h_{t-1}),\qquad c_t=0.5c_{t-1}+0.5g_t,\qquad h_t=0.5\tanh(c_t).$$

The mean command is $\mu_t=W_\pi h_t+b_\pi$. Initially $W_\pi=-0.5$, $b_\pi=0.02$ and $\sigma=e^{-1}$. The critic uses its own memory, adds $0.5w$ to its candidate input and predicts $V_t=0.8h_t^{V}+0.1$. These chosen initial weights make one forward pass checkable with a calculator.

In [2]:
policy = NativeActorCritic(obs_dim=2, actor_dim=1, action_dim=1, hidden_size=1,
                          actor_layers=(), critic_layers=(), log_std_init=-1.0)
with torch.no_grad():
    for parameter in policy.parameters():
        parameter.zero_()
    # PyTorch's gate order is input, forget, candidate, output.
    for lstm in (policy.actor_lstm, policy.critic_lstm):
        lstm.weight_ih_l0[2, 0] = 1.0
        lstm.weight_hh_l0[2, 0] = 0.5
    policy.critic_lstm.weight_ih_l0[2, 1] = 0.5
    policy.actor_head.weight.fill_(-0.5)
    policy.actor_head.bias.fill_(0.02)
    policy.critic_head.weight.fill_(0.8)
    policy.critic_head.bias.fill_(0.1)
    policy.log_std.fill_(-1.0)

first_c = 0.5 * np.tanh(0.4)
first_h = 0.5 * np.tanh(first_c)
print(f"First observation x=0.4, zero incoming memory: c={first_c:.5f}, "
      f"h={first_h:.5f}, mean action={-0.5*first_h + 0.02:.5f}")

First observation x=0.4, zero incoming memory: c=0.18997, h=0.09386, mean action=-0.02693


## 2. Collect the three decisions

The collector samples once per step and freezes the observations, raw actions, rewards, old values, old action log-densities and incoming memories. The first row starts with zero memory. Each later decision receives the memory produced by its predecessor. The table shows the outgoing actor memory $c_t,h_t$ after it reads $x_t$. The simulator receives the bounded command $u$ and returns a reward. PPO will later score the original Gaussian draw $a$. In these three rows all draws happen to lie inside the bounds. The weights stay fixed while the flight position and both networks’ memories change.

In [3]:
gamma, gae_lambda, clip_range = 0.99, 0.95, 0.2
torch.manual_seed(7)
collector = RolloutCollector(ThreeDecisionHover(), policy)
rollout = collector.collect(3, gamma, gae_lambda)
rng = np.random.default_rng(0)
batch = next(rollout.minibatches(batch_size=3, sequence_length=3, rng=rng))
assert batch.mask.all() and batch.episode_starts[:, 0].tolist() == [True, False, False]

# Revisit the rollout one decision at a time to expose the outgoing actor memory.
actor_hidden = []
state = batch.initial_state
with torch.no_grad():
    print(" t       x       c       h      mu       a       r   V_old  logp_old")
    for t in range(3):
        dist, value, state = policy(batch.obs[t:t+1], state, batch.episode_starts[t:t+1])
        h, c = state[0].item(), state[1].item()
        actor_hidden.append(h)
        print(f"{t+1:2} {batch.obs[t,0,0]:7.4f} {c:7.4f} {h:7.4f} "
              f"{dist.mean.item():7.4f} {batch.actions[t,0,0]:7.4f} "
              f"{rollout.rewards[t,0]:7.4f} {value.item():7.4f} {batch.old_log_prob[t,0]:9.4f}")

    mu_old = policy(batch.obs, batch.initial_state, batch.episode_starts)[0].mean.detach().clone()
    mu_reset = policy(batch.obs[1:2], policy.initial_state(1), batch.episode_starts[0:1])[0].mean
    print(f"At exactly the same second observation, mean with memory {mu_old[1].item():.5f}, "
          f"mean after a forced reset {mu_reset.item():.5f}.")
    assert not torch.allclose(mu_old[1], mu_reset[0])
np.testing.assert_allclose(actor_hidden[0], first_h, atol=1e-6)

# Follow the first draw from the Gaussian to the reward.
a = batch.actions[0, 0, 0].item()
u = np.clip(a, -1, 1)
x2 = 0.4 + 0.2 * (u + ThreeDecisionHover.wind)
r1 = 1 - abs(x2) - 0.05 * u**2
sigma = np.exp(-1.)
logp1 = -0.5 * ((a - mu_old[0].item()) / sigma)**2 - np.log(sigma * np.sqrt(2 * np.pi))
np.testing.assert_allclose([x2, r1, logp1],
                           [batch.obs[1, 0, 0], rollout.rewards[0, 0], batch.old_log_prob[0, 0]], atol=1e-6)
print(f"Draw a={a:.4f}, command u={u:.4f}, next x={x2:.4f}, reward={r1:.4f}, log density of a={logp1:.4f}")

 t       x       c       h      mu       a       r   V_old  logp_old
 1  0.4000  0.1900  0.0939 -0.0269 -0.0809  0.5759  0.1908    0.0703
 2  0.4238  0.3144  0.1522 -0.0561  0.2331  0.4868  0.2441   -0.2279
 3  0.5104  0.4209  0.1988 -0.0794  0.2689  0.3922  0.2820   -0.3672
At exactly the same second observation, mean with memory -0.05611, mean after a forced reset -0.02936.
Draw a=-0.0809, command u=-0.0809, next x=0.4238, reward=0.5759, log density of a=0.0703


## 3. Work backwards from the rewards

$V_t$ is the critic's estimate before the update. Compute the surprise at one step, $\delta_t=r_t+\gamma V_{t+1}-V_t$, then accumulate $A_t=\delta_t+\gamma\lambda A_{t+1}$ backwards. At the terminal step, $V_4=A_4=0$. A crash in Quadwind also has zero continuation. At a time limit, use the critic on the last observation before reset, then stop the GAE recursion. If a rollout fills mid-flight, use the next observation and the carried memory for the last TD error. The backward sum stops at the edge of the collected data. The critic target is $R_t=A_t+V_t$. The actor will use the standardized advantages $\hat A_t$ of the valid minibatch entries. A positive $\hat A_t$ favors increasing the density of this recorded action compared with its density under the collecting policy.

In [4]:
reward, value, next_value = (rollout.rewards[:, 0], rollout.values[:, 0], rollout.next_values[:, 0])
delta = reward + gamma * next_value - value
# Independent scalar calculation in terminal-to-first order.
a3 = reward[2] - value[2]
a2 = delta[1] + gamma * gae_lambda * a3
a1 = delta[0] + gamma * gae_lambda * a2
hand_adv = torch.stack((a1, a2, a3))
torch.testing.assert_close(hand_adv, batch.advantages[:, 0])
torch.testing.assert_close(value + hand_adv, batch.returns[:, 0])

print(" t   reward   V_old   V_next    delta       A   target")
for t in range(3):
    print(f"{t+1:2} {reward[t]:8.4f} {value[t]:7.4f} {next_value[t]:8.4f} "
          f"{delta[t]:8.4f} {hand_adv[t]:7.4f} {batch.returns[t,0]:8.4f}")
print(f"A3 = {reward[2]:.4f} - {value[2]:.4f} = {a3:.4f}")
print(f"A2 = {delta[1]:.4f} + {gamma} × {gae_lambda} × {a3:.4f} = {a2:.4f}")
print(f"First critic target = {value[0]:.4f} + {a1:.4f} = {batch.returns[0,0]:.4f}")

 t   reward   V_old   V_next    delta       A   target
 1   0.5759  0.1908   0.2441   0.6267  1.2150   1.4058
 2   0.4868  0.2441   0.2820   0.5219  0.6256   0.8696
 3   0.3922  0.2820   0.0000   0.1102  0.1102   0.3922
A3 = 0.3922 - 0.2820 = 0.1102
A2 = 0.5219 + 0.99 × 0.95 × 0.1102 = 0.6256
First critic target = 0.1908 + 1.2150 = 1.4058


## 4. Score the frozen flight with PPO

Replay the recorded actions from the original memory, in observation order. With $\rho_t=\exp(\log\pi_\theta(a_t)-\log\pi_{\mathrm{old}}(a_t))$, the objective for each valid decision is

$$s_t=\min\big(\rho_t\hat A_t,\ \mathrm{clip}(\rho_t,0.8,1.2)\hat A_t\big).$$

The actor loss is $-\mathrm{mean}(s_t)$. The critic loss is $\mathrm{mean}((V_t-R_t)^2)$. Here the total is the actor loss plus $0.5$ times the critic loss, matching the hover configuration's zero entropy coefficient. Before the first optimizer step both policies assign the same density to each action, so every $\rho_t=1$. Standardized advantages average to zero here, making the initial actor loss zero. The recorded actions are constants. Their log-densities depend on the current weights, so their contributions can have a nonzero gradient even when the actor loss averages to zero.

In [5]:
def score():
    return ppo_loss(policy, batch, clip_range=clip_range,
                    vf_coef=0.5, ent_coef=0.0, normalize_advantage=True)

loss = score()
adv = batch.advantages[:, 0]
standard_adv = (adv - adv.mean()) / (adv.std() + 1e-8)
torch.testing.assert_close(loss.advantages.flatten(), standard_adv)
torch.testing.assert_close(loss.ratio.flatten(), torch.ones_like(adv))
torch.testing.assert_close(loss.value, (value - batch.returns[:, 0]).square().mean())
print("Standardized A =", [round(v, 4) for v in standard_adv.tolist()])
print("Density ratios =", loss.ratio.flatten().detach().tolist())
print(f"Actor loss {loss.policy.item():.6f}   critic MSE {loss.value.item():.6f}   "
      f"total {loss.total.item():.6f}")

Standardized A = [1.0216, -0.0447, -0.9769]
Density ratios = [1.0, 1.0, 1.0]
Actor loss 0.000000   critic MSE 0.626598   total 0.313299


## 5. Predict one weight change, then take one Adam step

Track the actor's output weight $W_\pi$, the multiplier of $h_t$. At the old policy all ratios equal 1. The Gaussian log-density derivative gives the following local gradient of the actor loss.

$$g_{\rm raw}=-\frac{1}{3}\sum_{t=1}^3 \hat A_t\,\frac{a_t-\mu_t}{\sigma^2}\,h_t.$$

The trainer clips the norm of **all** gradients together at $0.5$. Call the resulting gradient for this weight $g$. Adam starts with zero moments, so its first bias-corrected moments are $g$ and $g^2$. Thus $W'_\pi=W_\pi-\eta g/(|g|+\epsilon)$. We enlarge the example learning rate so the movement is visible on three decisions. The full hover run starts at $0.0003$.

In [6]:
learning_rate = 0.08
optimizer = torch.optim.Adam(policy.parameters(), lr=learning_rate, eps=1e-5)
weight = policy.actor_head.weight
weight_before = weight.item()
sigma_sq = policy.log_std.detach().exp().square().item()
hidden = torch.tensor(actor_hidden, dtype=adv.dtype)
terms = -standard_adv * (batch.actions[:, 0, 0] - mu_old[:, 0, 0]) * hidden / sigma_sq
manual_grad = terms.mean().item()
# Predict the global clipping factor before changing any weights.
parameters = dict(policy.named_parameters())
gradients = torch.autograd.grad(loss.total, parameters.values(), retain_graph=True)
total_norm = sum(grad.square().sum().item() for grad in gradients) ** 0.5
autograd_grad = dict(zip(parameters, gradients))["actor_head.weight"].item()
np.testing.assert_allclose(manual_grad, autograd_grad, atol=1e-6)

g = manual_grad * min(1.0, 0.5 / (total_norm + 1e-6))
expected_weight = weight_before - learning_rate * g / (abs(g) + optimizer.param_groups[0]["eps"])
measured_norm = float(optimizer_step(policy, optimizer, loss, max_grad_norm=0.5))
np.testing.assert_allclose(total_norm, measured_norm, atol=1e-6)
np.testing.assert_allclose(g, weight.grad.item(), atol=1e-6)
np.testing.assert_allclose(expected_weight, weight.item(), atol=1e-6)
print("Three gradient contributions =", [round(x, 6) for x in terms.tolist()])
print(f"Raw gradient = {manual_grad:.6f}   global norm = {total_norm:.6f}   "
      f"gradient used by Adam = {g:.6f}")
print(f"W_old {weight_before:.6f}   hand prediction {expected_weight:.6f}   "
      f"after optimizer.step() {weight.item():.6f}")

Three gradient contributions = [0.038262, 0.014528, 0.499953]
Raw gradient = 0.184248   global norm = 1.287316   gradient used by Adam = 0.071563
W_old -0.500000   hand prediction -0.579989   after optimizer.step() -0.579989


## 6. What changed on these exact observations?

Evaluate the **same actions**, from the **same initial memories**, with the updated weights. Check where the mean command moved and whether the critic moved toward its fixed target. The last two columns show the raw and selected objective contribution on a possible second pass through these data. This second evaluation does not take another optimizer step.

In [7]:
with torch.no_grad():
    after = score()
    new_dist, new_values, _ = policy(batch.obs, batch.initial_state, batch.episode_starts)
    ratios = after.ratio.flatten()
    raw = ratios * standard_adv
    capped = ratios.clamp(1.0-clip_range, 1.0+clip_range) * standard_adv
    selected = torch.minimum(raw, capped)
    torch.testing.assert_close(selected, after.surrogate.flatten())
    old_logp = batch.old_log_prob[2, 0].item()
    new_logp = new_dist.log_prob(batch.actions)[2, 0].item()
    print(f"Decision 3 density ratio = exp({new_logp:.4f} - ({old_logp:.4f})) = {ratios[2]:.4f}")
    print(" t   mu_old   mu_new    V_old    V_new   target    ratio    rho*A   PPO term")
    for t in range(3):
        print(f"{t+1:2} {mu_old[t].item():8.4f} {new_dist.mean[t].item():8.4f} "
              f"{value[t]:8.4f} {new_values[t].item():8.4f} {batch.returns[t,0]:8.4f} "
              f"{ratios[t]:8.4f} {raw[t]:8.4f} {selected[t]:10.4f}")
    print(f"Critic MSE on this flight  {loss.value.item():.4f} → {after.value.item():.4f}")
    assert after.value < loss.value and capped[2] < raw[2]
    for t in range(3):
        if capped[t] < raw[t]:
            print(f"Decision {t+1}: standardized A={standard_adv[t]:.4f}, ratio={ratios[t]:.4f}. "
                  f"Clipping selects {selected[t]:.4f} instead of the raw {raw[t]:.4f}.")

Decision 3 density ratio = exp(-0.9903 - (-0.3672)) = 0.5362
 t   mu_old   mu_new    V_old    V_new   target    ratio    rho*A   PPO term
 1  -0.0269  -0.1500   0.1908   0.3374   1.4058   1.0726   1.0958     1.0958
 2  -0.0561  -0.2068   0.2441   0.4310   0.8696   0.6375  -0.0285    -0.0357
 3  -0.0794  -0.2464   0.2820   0.4901   0.3922   0.5362  -0.5239    -0.7815
Critic MSE on this flight  0.6266 → 0.4479
Decision 2: standardized A=-0.0447, ratio=0.6375. Clipping selects -0.0357 instead of the raw -0.0285.
Decision 3: standardized A=-0.9769, ratio=0.5362. Clipping selects -0.7815 instead of the raw -0.5239.


### What to say at the defense

On decision 1, the new policy shifts its mean from **−0.0269 to −0.1500**, in the direction that pushes back against the wind. The critic moves **0.1908 → 0.3374** toward a target of **1.4058**. The evidence here is an update on one flight. The held-out benchmark measures how well a policy flies.

1. Memory $(h,c)$ changes with each observation while the aircraft flies. A fresh episode resets it. During training, replaying a sequence lets the loss send gradients through the LSTM.
2. The final reward determines the last advantage directly. GAE carries part of that signal back through the two earlier decisions. The critic learns toward the resulting targets.
3. The last action has the smallest advantage. After standardization its $\hat A_3$ is **−0.9769**, so reducing this action's density improves its surrogate. At $\rho_3=0.5362$, the raw contribution is −0.5239. PPO selects $0.8\times(-0.9769)=-0.7815$. Further reductions of this ratio give no extra gain on that term. For a positive advantage of +1, a ratio of 1.5 would contribute 1.2, while a ratio of 0.5 would contribute 0.5. The ratio itself can fall below 0.8.
4. A full hover rollout contains 16 environments × 512 steps. Its actor and critic each have 128 LSTM units. It reuses ordered sequence minibatches for five epochs, with a budget of 3 million environment steps. Here we used three steps and one update so every number can be inspected. Flight performance needs separate held-out evaluations. The previously recorded E7 hover scores belong to the SB3-Contrib runs.

### Optional check for questions · Recompute the updated critic prediction

At the first observation the critic starts with $h_0=c_0=0$. Read the four updated gate preactivations from $z=W_{ih}[x_1,w]+b_{ih}+b_{hh}$ in PyTorch's input, forget, candidate, output order. The recurrent-weight term is zero here. Adam updated the gates as well as the value head, so we evaluate their new values. With $i=\sigma(z_i)$, $g=\tanh(z_g)$ and $o=\sigma(z_o)$, compute $c_1=i g$, $h_1=o\tanh(c_1)$, then $V_1'=W_V'h_1+b_V'$. The forget gate multiplies zero initial cell memory.

In [8]:
# One scalar cell, using only the updated weights and NumPy arithmetic.
lstm = policy.critic_lstm
x1 = batch.obs[0, 0].numpy()
gate_input = lstm.weight_ih_l0.detach().numpy() @ x1
z = gate_input + lstm.bias_ih_l0.detach().numpy() + lstm.bias_hh_l0.detach().numpy()
i, f, o = 1.0 / (1.0 + np.exp(-z[[0, 1, 3]]))
candidate = np.tanh(z[2])
c1 = i * candidate
h1 = o * np.tanh(c1)
w_v = policy.critic_head.weight.item()
b_v = policy.critic_head.bias.item()
v1_hand = w_v * h1 + b_v
np.testing.assert_allclose(v1_hand, new_values[0, 0].item(), atol=1e-6)
print(f"Updated gates  i={i:.5f}  f={f:.5f}  candidate={candidate:.5f}  o={o:.5f}")
print(f"c1 = {i:.5f} × {candidate:.5f} = {c1:.5f}    h1 = {o:.5f} × tanh({c1:.5f}) = {h1:.5f}")
print(f"V1 by hand = {w_v:.5f} × {h1:.5f} + {b_v:.5f} = {v1_hand:.5f}    network = {new_values[0,0]:.5f}")

Updated gates  i=0.55176  f=0.55147  candidate=0.60937  o=0.55176
c1 = 0.55176 × 0.60937 = 0.33623    h1 = 0.55176 × tanh(0.33623) = 0.17883
V1 by hand = 0.87998 × 0.17883 + 0.18000 = 0.33736    network = 0.33736
